In [1]:
import sys
sys.path.insert(1,r'C:\Users\krajcovic\Documents\Algo\BorderSpread_tools')
import numpy as np
import pandas as pd
#from BorderSpread_tools import *
import BorderSpread.border_class as bs
import BorderSpread.capacity_class as capa
from BorderSpread.PositionManager import PositionManagerCapa, PositionManagerHedge
from BorderSpread.capacity_contract_class import CapacityContr, FwdContract
from Loaders.EikonSpot_class import EikonSpot as es
import datetime as dt
from dateutil.relativedelta import relativedelta
import calendar
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
fut_codes = pd.read_excel(r'C:\Users\krajcovic\Documents\Trading\Data\futures codes.xlsx',
                         sheet_name='countries')
month_codes = pd.read_excel(r'C:\Users\krajcovic\Documents\Trading\Data\futures codes.xlsx',
                         sheet_name='months')

In [23]:
capa_fv_list = []
delta_list = []
spread_list = []
fut1_list = []
fut2_list = []


border_list = ['de_at', 'at_de', 'at_hu', 'hu_at',
              'cz_sk', 'sk_cz', 'sk_hu', 'hu_sk',
              'de_cz', 'cz_de', 'at_cz', 'cz_at']

    


base_today = dt.date.today()
month_start = dt.date(base_today.year,
                      base_today.month,
                      1)


for border in border_list:
    border = [border]
    bs_class = bs.DataBorderClass(border,
                                 ['implicit'],
                                 start_date=dt.datetime(2023,3,1),
                                 end_date=dt.date.today()-dt.timedelta(days=-1,hours=1))
    #bs_class.load_data(path_spot=r'C:\Users\krajcovic\Documents\Trading\Data\Price\EEX\Spot\spot.txt')
    bs_class.load_data()
    delivery_ = ['base']
    data = dict()
    for del_ in delivery_:
        data[del_] = bs_class.aggregate_data('W', delivery=del_)

    capacity = capa.ImplicitCapacity(border, delivery_)
    for b in border:
        for del_ in delivery_:
            capacity.capa_fit(data[del_], del_, scaling=4)
    month = 10
    year = 2023

    fut_o1 = es(border[0].lower().split('_')[0],
               month_start.strftime('%Y-%m-%d'),
               base_today.strftime('%Y-%m-%d'))
    fut_o2 = es(border[0].lower().split('_')[1],
               month_start.strftime('%Y-%m-%d'),
               base_today.strftime('%Y-%m-%d'))
        
    fut1 = fut_o1.fwd_df_rel(fut_o1.start_date,
                      fut_o1.end_date,
                      ['M_1'], ['base']).iloc[-1]
    fut2 = fut_o2.fwd_df_rel(fut_o2.start_date,
                      fut_o2.end_date,
                      ['M_1'], ['base']).iloc[-1]
    
    
    spread = float(fut1) - float(fut2)
    
    delta =  capacity.capa_delta(float(fut1),
                          float(fut2),
                          'base')[1]
    capa_fv = capacity.capa_price(float(fut1),
                          float(fut2),
                          'base')[1]
    
    capa_fv_list.append(capa_fv)
    delta_list.append(delta)
    spread_list.append(spread)

Statement 'SELECT datetime,price FROM spot.de WHERE datetime BETWEEN '2023-03-01 00:00:00' AND '2023-11-24 23:59:00' ORDER BY datetime ASC' executed
Statement 'SELECT datetime,price FROM spot.fr WHERE datetime BETWEEN '2023-03-01 00:00:00' AND '2023-11-24 23:59:00' ORDER BY datetime ASC' executed
Statement 'SELECT datetime,price FROM spot.fr WHERE datetime BETWEEN '2023-03-01 00:00:00' AND '2023-11-24 23:59:00' ORDER BY datetime ASC' executed
Statement 'SELECT datetime,price FROM spot.de WHERE datetime BETWEEN '2023-03-01 00:00:00' AND '2023-11-24 23:59:00' ORDER BY datetime ASC' executed
Statement 'SELECT datetime,price FROM spot.de WHERE datetime BETWEEN '2023-03-01 00:00:00' AND '2023-11-24 23:59:00' ORDER BY datetime ASC' executed
Statement 'SELECT datetime,price FROM spot.be WHERE datetime BETWEEN '2023-03-01 00:00:00' AND '2023-11-24 23:59:00' ORDER BY datetime ASC' executed
Statement 'SELECT datetime,price FROM spot.be WHERE datetime BETWEEN '2023-03-01 00:00:00' AND '2023-11-24

In [14]:
delta

-0.7844392571241247

In [24]:
fv_df = pd.DataFrame([border_list,
                     capa_fv_list,
                     delta_list,
                     spread_list]).T
fv_df.columns = ['border', 'fv', 'delta', 'spread']
fv_df['spread'] = fv_df['spread']*(-1)
fv_df['ext'] = np.where(fv_df['spread']>0,
                        fv_df['fv']-fv_df['spread'],
                        fv_df['fv'])

In [26]:
fv_df

,border,fv,delta,spread,ext
0,de_fr,9.750458,0.629515,4.37,5.380458
1,fr_de,5.380458,0.44335,-4.37,5.380458
2,de_be,9.038288,0.670464,5.44,3.598288
3,be_de,3.598289,0.385656,-5.44,3.598289
4,de_nl,7.749538,0.698033,5.32,2.429538
5,nl_de,2.429538,0.343983,-5.32,2.429538
6,be_nl,4.622179,0.519006,-0.12,4.622179
7,nl_be,4.742179,0.527146,0.12,4.622179
8,be_fr,2.889231,0.466759,-1.07,2.889231
9,fr_be,3.959231,0.566627,1.07,2.889231


In [22]:
fv_df

,border,fv,delta,spread,ext
0,de_fr,6.941747,0.673958,4.37,2.571747
1,fr_de,2.571747,0.367647,-4.37,2.571747
2,de_be,7.014614,0.738117,5.44,1.574614
3,be_de,1.574614,0.293106,-5.44,1.574614
4,de_nl,6.452032,0.766016,5.32,1.132032
5,nl_de,1.132032,0.258727,-5.32,1.132032
6,be_nl,3.413961,0.511629,-0.12,3.413961
7,nl_be,3.533961,0.522611,0.12,3.413961
8,be_fr,1.467987,0.423144,-1.07,1.467987
9,fr_be,2.537987,0.595779,1.07,1.467987


In [19]:
fv_df

,border,fv,delta,spread,ext
0,de_fr,5.981184,0.713146,4.37,1.611184
1,fr_de,1.611184,0.316676,-4.37,1.611184
2,de_be,6.304217,0.792936,5.44,0.864217
3,be_de,0.864217,0.227787,-5.44,0.864217
4,de_nl,5.892823,0.826793,5.32,0.572823
5,nl_de,0.572823,0.188762,-5.32,0.572823
6,be_nl,2.249598,0.503113,-0.12,2.249598
7,nl_be,2.369598,0.519645,0.12,2.249598
8,be_fr,1.136269,0.403532,-1.07,1.136269
9,fr_be,2.206269,0.611917,1.07,1.136269


In [16]:
fv_df

,border,fv,delta,spread,ext
0,de_fr,5.978937,-0.683324,4.37,1.608937
1,fr_de,1.608937,-0.286854,-4.37,1.608937
2,de_be,6.302778,-0.772213,5.44,0.862778
3,be_de,0.862778,-0.207064,-5.44,0.862778
4,de_nl,5.891796,-0.811238,5.32,0.571796
5,nl_de,0.571796,-0.173207,-5.32,0.571796
6,be_nl,2.248413,-0.480355,-0.12,2.248413
7,nl_be,2.368413,-0.496887,0.12,2.248413
8,be_fr,1.135745,-0.388083,-1.07,1.135745
9,fr_be,2.205745,-0.596468,1.07,1.135745


In [11]:
fv_df

,border,fv,delta,spread,ext
0,de_fr,1.716067,0.351647,-3.26,1.716067
1,fr_de,4.976067,0.678574,3.26,1.716067
2,dke_de,21.330221,0.999859,21.33,0.000221
3,de_dke,0.000202,0.00017,-21.33,0.000202
4,at_de,0.081696,0.057824,-5.49,0.081696
5,de_at,5.571696,0.946157,5.49,0.081696
6,be_de,2.868191,0.536682,0.38,2.488191
7,de_be,2.488191,0.491558,-0.38,2.488191
8,be_fr,0.450062,0.221576,-2.88,0.450062
9,fr_be,3.330062,0.789951,2.88,0.450062


In [14]:
fv_df

,border,fv,delta,spread,ext
0,de_fr,1.716067,0.351647,-3.26,1.716067
1,fr_de,4.976067,0.678574,3.26,1.716067
2,dke_de,21.330221,0.999859,21.33,0.000221
3,de_dke,0.000202,0.00017,-21.33,0.000202
4,at_de,0.081696,0.057824,-5.49,0.081696
5,de_at,5.571696,0.946157,5.49,0.081696
6,be_de,2.868191,0.536682,0.38,2.488191
7,de_be,2.488191,0.491558,-0.38,2.488191
8,be_fr,0.450062,0.221576,-2.88,0.450062
9,fr_be,3.330062,0.789951,2.88,0.450062


In [31]:
fut_o1 = es('at',
           month_start.strftime('%Y-%m-%d'),
           base_today.strftime('%Y-%m-%d'))
fut_o2 = es('de',
           month_start.strftime('%Y-%m-%d'),
           base_today.strftime('%Y-%m-%d'))
    


In [32]:
fut1 = fut_o1.fwd_df_rel(fut_o1.start_date,
                  fut_o1.end_date,
                  ['M_1'], ['base'])
fut2 = fut_o2.fwd_df_rel(fut_o2.start_date,
                  fut_o2.end_date,
                  ['M_1'], ['base'])


spread = fut1 - fut2

In [33]:
spread

,M_1
2023-07-30,NaN
2023-07-31,3.25
2023-08-01,5.75
2023-08-02,5.75
2023-08-03,4.30
2023-08-04,4.25
2023-08-05,4.25
2023-08-06,4.25
2023-08-07,4.63
2023-08-08,3.75


In [50]:
capa_fv_list = []
delta_list = []
fut1_list = []
fut2_list = []
spread_list = []
border_list =['at_de', 'de_at', 'fr_de', 'de_fr']


base_today = dt.date.today()
month_start = dt.date(base_today.year,
                      base_today.month,
                      1)


for border in border_list:
    border = [border]
    bs_class = bs.DataBorderClass(border,
                                 ['implicit'],
                                 start_date=dt.datetime(2023,1,1),
                                 end_date=dt.date.today()-dt.timedelta(days=-1,hours=1))
    #bs_class.load_data(path_spot=r'C:\Users\krajcovic\Documents\Trading\Data\Price\EEX\Spot\spot.txt')
    bs_class.load_data()
    delivery_ = ['base']
    data = dict()
    for del_ in delivery_:
        data[del_] = bs_class.aggregate_data('W', delivery=del_)

    capacity = capa.ImplicitCapacity(border, delivery_)
    for b in border:
        for del_ in delivery_:
            capacity.capa_fit(data[del_], del_)
    month = 9
    year = 2023

    
    if border[0] == 'de_fr':
        fut1 = 104.25
        fut2 = 101.75
    else:
        fut2 = 104.25
        fut1 = 101.75
    spread = float(fut1) - float(fut2)
    
    delta =  capacity.capa_delta(float(fut1),
                          float(fut2),
                          'base')[0]
    capa_fv = capacity.capa_price(float(fut1),
                          float(fut2),
                          'base')[0]
    
    capa_fv_list.append(capa_fv)
    delta_list.append(delta)
    spread_list.append(spread)

In [51]:
fv_df = pd.DataFrame([border_list,
                     capa_fv_list,
                     delta_list,
                     spread_list]).T
fv_df.columns = ['border', 'fv', 'delta', 'spread']
fv_df['spread'] = fv_df['spread']*(-1)
fv_df['ext'] = np.where(fv_df['spread']>0,
                        fv_df['fv']-fv_df['spread'],
                        fv_df['fv'])

In [52]:
fv_df

,border,fv,delta,spread,ext
0,at_de,3.032175,0.759245,2.5,0.532175
1,de_at,3.032175,0.759245,2.5,0.532175
2,fr_de,4.793895,0.630975,2.5,2.293895
3,de_fr,2.293896,0.400675,-2.5,2.293896
